# 11 - Build the Q1 LLM Benchmark (300 profiles)

Creates the **frozen** held-out benchmark for the final six-arm ablation.

```text
Routine lifestyle   90   (30 / 30 / 30)
Comorbidity         60   (20 / 20 / 20)
Emergency/red flag  60   (20 / 20 / 20)
Boundary/uncertain  30   (10 / 10 / 10)
Missing input       30   (10 / 10 / 10)
Adversarial         30   (10 / 10 / 10)
                   300   (100 low / 100 moderate / 100 high)
```

Risk category, probabilities, confidence, `key_model_factors` and the reliability
band all come from calling the **real deployed cascade** on each sampled payload -
nothing is hand-labelled. Cells are filled by rejection sampling on the model's
actual output.

**Runtime:** CPU only. No GPU required.

> Once built and hashed, this benchmark must not be modified after any model
> output has been observed.


In [ ]:
# ============================================================
# Cell 1 - Environment and project code
# ============================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = ''                      # <-- set to your git remote, or upload the repo
REPO = Path('/content/Nephro-AI') if Path('/content').exists() else Path.cwd()

if not (REPO / 'llm').exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
    else:
        raise SystemExit('Set REPO_URL or place the project at ' + str(REPO))

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('cwd:', Path.cwd())


In [ ]:
# ============================================================
# Cell 2 - Dependencies (CPU only)
# ============================================================
%pip install -q 'scikit-learn==1.6.1' 'joblib==1.4.2' 'pandas==2.2.3' 'numpy==2.0.2' 'scipy==1.14.1' matplotlib

from backend.model_service import KidneyRiskModelService
svc = KidneyRiskModelService()
print('model loaded:', svc.loaded, '| features:', len(svc.model_features))
assert svc.loaded, svc.load_error


## 2. Build

Each group/class cell is filled by rejection sampling. If a cell stalls, the
sampler relaxes its optional shaping in steps - but never its defining property
(emergency profiles always keep >=180/120 vitals; missing-input profiles always
keep missing fields). Any relaxation used is recorded in the profile card.


In [ ]:
# ============================================================
# Cell 3 - Generate the 300 profiles (deterministic, seed 42)
# ============================================================
from llm.eval.q1 import benchmark as bm

profiles, audit, relaxations = bm.build_benchmark(seed=42)
print(f'\nbuilt {len(profiles)} profiles')
print('relaxations used:', relaxations or 'none')


In [ ]:
# ============================================================
# Cell 4 - Distribution
# ============================================================
import pandas as pd
from collections import Counter

df = pd.DataFrame([{
    'group': p['group'], 'risk': p['risk_class'],
    'band': p['reliability_band'],
    'red_flags': len(p['context']['red_flags']),
    'missing': len(p['context']['missing_fields']),
    'attack': p['evaluator']['attack_category'] or '',
} for p in profiles])

display(pd.crosstab(df['group'], df['risk'], margins=True))
print('\nreliability bands:', dict(Counter(df['band'])))
print('with red flags   :', int((df['red_flags'] > 0).sum()))
print('with missing     :', int((df['missing'] > 0).sum()))
print('attack categories:', dict(Counter(a for a in df['attack'] if a)))


## 3. Duplicate audit

Every candidate is compared against the 600 Phase-7 fine-tuning examples, the 60
validation examples and the 30 Phase-6 profiles - by exact 14-feature signature
and by an age-excluded signature, so a profile that differs *only* by age is
rejected as a near duplicate.


In [ ]:
# ============================================================
# Cell 5 - Duplicate audit summary
# ============================================================
adf = pd.DataFrame(audit)
print('candidates examined :', len(adf))
print('accepted            :', int((adf['resolution'] == 'accepted').sum()))
print('rejected as dupes   :', int((adf['resolution'] == 'resampled').sum()))
if (adf['resolution'] == 'resampled').any():
    display(adf[adf['resolution'] == 'resampled']['duplicate_kind'].value_counts())


## 4. Integrity gate

Artifacts are only written if **every** check passes.


In [ ]:
# ============================================================
# Cell 6 - PASS/FAIL integrity table
# ============================================================
checks = bm.verify_benchmark(profiles)
w = max(len(c[0]) for c in checks)
all_ok = True
print('=== PART 1 INTEGRITY TABLE ===')
for name, ok, detail in checks:
    all_ok &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  {name.ljust(w)}  {detail}")
print()
print('OVERALL:', 'PASS' if all_ok else 'FAIL')
assert all_ok, 'Benchmark verification failed - artifacts not written.'


In [ ]:
# ============================================================
# Cell 7 - Write + freeze the artifacts
# ============================================================
paths = bm.write_artifacts(profiles, audit, Path('q1_benchmark'), 42, relaxations)
print('BENCHMARK HASH:', paths['benchmark_hash'])
for k, v in paths.items():
    if k != 'benchmark_hash':
        print(f'  {k:16s} -> {v}')

print()
print(Path('q1_benchmark/q1_profile_card.md').read_text(encoding='utf-8')[:2500])


## 5. Freeze

Record the benchmark hash printed above in the run manifest and the paper.
**Do not regenerate or edit these files after any arm has been run.**

Next: `12_RUN_Q1_LLM_SIX_ARM_ABLATION.ipynb`.
